# Fitting the Single-Diode Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aiyush-G/semiconductor-diode-fitting/blob/main/explanations/models/02_single_diode_fitting.ipynb)

**An interactive walkthrough of `src/models/fitting.py`**


---

### Contents


|  | | |
|---|---|---|
| **1** | What is a J–V curve? |
| **2** | The equivalent circuit |  
| **3** | The inverse problem |
| **4** | How `fitting.py` is built |
| **5** | **Degeneracy** |
| **6** | Assumptions and gaps |
| **7** | References |

### How to run

`Runtime → Run all` (or ⌘/Ctrl+F9). Takes about a minute.

## 0 · Setup

The cell below finds the repo.

In [1]:
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/Aiyush-G/semiconductor-diode-fitting.git"
IN_COLAB = "google.colab" in sys.modules


def find_root(start: pathlib.Path) -> pathlib.Path | None:
    """Walk up from `start` looking for the repo root (the dir containing src/models)."""
    for p in [start, *start.parents]:
        if (p / "src" / "models" / "fitting.py").exists():
            return p
    return None


root = find_root(pathlib.Path.cwd())
if root is None:
    dest = pathlib.Path("/content" if IN_COLAB else ".") / "semiconductor-diode-fitting"
    if not (dest / "src").exists():
        print(f"Cloning {REPO_URL} -> {dest}")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True
        )
    root = dest

root = root.resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Repository root: {root}")

Cloning https://github.com/Aiyush-G/semiconductor-diode-fitting.git -> /content/semiconductor-diode-fitting
Repository root: /content/semiconductor-diode-fitting


In [2]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# The physics + fitting core (numpy/scipy only).
import src.models.fitting as fitting_module
from src.models.examples import DARK_JV_EXAMPLE, LIGHT_JV_EXAMPLE
from src.models.fitting import (
    DEFAULT_BOUNDS,
    PARAM_NAMES,
    default_specs,
    fit_diode,
    pack,
    resolve_residual_space,
)
from src.models.single_diode import (
    DiodeParams,
    key_metrics,
    solve_current,
    thermal_voltage,
)

# The app's own figure builders
from ui.plotting import iv_curve_figure, log_jv_figure, residual_figure

# Colab renders plotly out of the box; this only matters for other frontends.
if IN_COLAB:
    pio.renderers.default = "colab"

# The whole reuse argument above rests on this being true:
assert "streamlit" not in sys.modules, "ui.plotting should not drag in streamlit"

print("fitting.py  :", fitting_module.__file__)
print("numpy       :", np.__version__)
print("no streamlit: OK")

fitting.py  : /content/semiconductor-diode-fitting/src/models/fitting.py
numpy       : 2.0.2
no streamlit: OK


Interactive controls are `ipywidgets` sliders.

In [3]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False

print("interactive widgets:", "available" if HAVE_WIDGETS else "unavailable (static fallback)")

interactive widgets: available


---
## 1 · J–V curve


Shine standardised light on solar cell (**1 sun** = 100 mW/cm², roughly midday),
and then **sweep the voltage** across its terminals while **measuring the current** that comes out.

In [4]:
ds = LIGHT_JV_EXAMPLE
metrics = key_metrics(ds.voltage, ds.current)

fig = iv_curve_figure(
    ds.voltage, ds.current, metrics=metrics, title="J–V curve Example",
)
fig.show()

print(f"{ds.label}: {ds.voltage.size} points, {ds.voltage.min():.3f}–{ds.voltage.max():.3f} V")

Example: Light JV: 64 points, 0.000–0.613 V


### Reading JV Curve

The blue curve is current; the faint
second trace is **power** ($P = V \times J$), which is what needs to be maximised.

- **$J_{sc}$ - short-circuit current density.** Current at $V = 0$.
- **$V_{oc}$ - open-circuit voltage.** The voltage where current falls to zero. No current
  flows, so no power is delivered.
- **MPP - maximum power point.**
- **Fill factor** - $P_{max} / (J_{sc}V_{oc})$: how *square* the curve is. A perfect cell would hold
  $J_{sc}$ right up to $V_{oc}$ (FF = 1); resistance rounds the knee of the graph.

`key_metrics()` extracts these parameters (see below)

In [5]:
print(f"Jsc         = {metrics['jsc'] * 1e3:7.2f} mA/cm^2   (current at V = 0)")
print(f"Voc         = {metrics['voc']:7.4f} V        ( current hits zero)")
print(f"Pmax        = {metrics['pmax'] * 1e3:7.2f} mW/cm^2   (at the knee)")
print(f"Fill factor = {metrics['fill_factor']:7.3f}          (square the curve is)")
print(f"Efficiency  = {metrics['efficiency'] * 1e2:7.2f} %        (Pmax / 100 mW/cm^2 of sunlight)")

Jsc         =   36.06 mA/cm^2   (current at V = 0)
Voc         =  0.6127 V        ( current hits zero)
Pmax        =   17.03 mW/cm^2   (at the knee)
Fill factor =   0.771          (square the curve is)
Efficiency  =   17.03 %        (Pmax / 100 mW/cm^2 of sunlight)


---
## 2 · The equivalent circuit


**1.Current source.** Light knocks electrons loose at a steady rate, so the cell generates a
constant photocurrent $J_{ph}$ regardless of voltage:

$$ J = J_{ph} $$

A flat line. Infinite power at high voltage - this is quite incorrect. Therefore:

**2. A diode, in parallel.** A solar cell *is* the pn jn / diode. As voltage is increased,
the junction starts conducting backwards through the cell, reducing the photocurrent. This is
recombination — where carriers anhillate each other. It grows exponentially:

$$ J = J_{ph} - J_0\left(e^{V/nV_t} - 1\right) $$

- $J_0$ - **saturation current density**: how leaky the junction is. Smaller is better. It is tiny
  (~$10^{-11}$) but sits under an exponential, so it dominates $V_{oc}$.
- $n$ - **ideality factor**: *how* recombination happens. $n = 1$ is band-to-band; $n = 2$ means
  defect-mediated (Shockley–Read–Hall). Between 1 and 2 in practice.
- $V_t = k_BT/q$ - **thermal voltage**

Introduce defects into the model:

**3. Shunt resistance $R_{sh}$, in parallel.** Manufacturing defects give current a short-cut
straight across the junction. Ideally, R is infinite.

$$ J = J_{ph} - J_0\left(e^{V/nV_t} - 1\right) - \frac{V}{R_{sh}} $$

**4. A series resistance $R_s$, in series.** It should be small (ideally zero). Junction doesn't see $V$ but rather it sees $V + JR_s$:

$$ J = J_{ph} - J_0\left(e^{\frac{V + JR_s}{nV_t}} - 1\right) - \frac{V + JR_s}{R_{sh}} $$


$J$ is on both sides

---



FUll equation:

$$ J = J_{ph} - J_0\left(e^{\frac{V + JR_s}{nV_t}} - 1\right) - \frac{V + JR_s}{R_{sh}} $$

$J$ appears on the left and inside the exponential on the right. It is implicit.

Thus use the standard closed form (Jain & Kapoor) based on the **Lambert W function** -
the inverse of $w \mapsto we^{w}$, which is exactly the shape "unknown inside an exponential *and*
outside it" produces. `solve_current` computes:

$$ a = \frac{R_sR_{sh}J_0}{nV_t(R_s+R_{sh})}, \qquad
   b = \frac{R_{sh}\big(R_s(J_{ph}+J_0)+V\big)}{nV_t(R_s+R_{sh})} $$

$$ J = \frac{R_{sh}(J_{ph}+J_0) - V}{R_s+R_{sh}} - \frac{nV_t}{R_s}\,W\!\left(a\,e^{b}\right) $$


Note $R_s = 0$  and means can't divide by zero


---
## 3 · Fitting (basically the inverse of the above)

Everything so far runs the physics forwards: five parameters, get a curve.

$$ (J_{ph},\, J_0,\, n,\, R_s,\, R_{sh}) \;\longrightarrow\; J(V) $$

But now...

$$ J(V)_{\text{measured}} \;\longrightarrow\; (J_{ph},\, J_0,\, n,\, R_s,\, R_{sh})\;? $$

"How wrong" each guess is needs a number. Use difference at every measured point (the **residual**), square
it, average, square-root - the **RMSE**.

Now let the module do it. Same data, same five parameters — but it searches systematically.

In [32]:
specs = default_specs("light", free={"j_ph", "j_0", "n", "r_s", "r_sh"})
result = fit_diode(DS.voltage, DS.current, TEMP_K, specs, kind="light")

p = result.params
print(f"converged : {result.success}  ({result.message})")
print(f"free      : {', '.join(result.free_names)}\n")
print(f"  Jph  = {p.j_ph * 1e3:.4f} mA/cm^2")
print(f"  J0   = {p.j_0:.4e} A/cm^2")
print(f"  n    = {p.n:.4f}")
print(f"  Rs   = {p.r_s:.4f} Ohm.cm^2")
print(f"  Rsh  = {p.r_sh:.2f} Ohm.cm^2\n")
print(f"  RMSE = {result.rmse * 1e3:.4f} mA/cm^2")
print(f"  R^2  = {result.r_squared:.7f}")

converged : True  (`gtol` termination condition is satisfied.)
free      : j_ph, j_0, n, r_s, r_sh

  Jph  = 36.0926 mA/cm^2
  J0   = 4.5802e-11 A/cm^2
  n    = 1.1669
  Rs   = 0.2932 Ohm.cm^2
  Rsh  = 437.18 Ohm.cm^2

  RMSE = 0.0452 mA/cm^2
  R^2  = 0.9999618


R² = 0.99996 but needs validating for meaning

---
## 4 · Explanation of `fitting.py`


Formally, the module minimises a sum of squared residuals over the free parameters $\theta$:

$$ \hat\theta \;=\; \arg\min_{\theta \in [\ell,\,u]} \; \tfrac12\sum_{i=1}^{m} r_i(\theta)^2 $$

Three properties of this problem drive everything below:

1. **It is nonlinear**
2. **The parameters of diff magnitudes** $J_0 \approx 10^{-11}$ and
   $R_{sh} \approx 10^{3}$ differ by *fourteen orders of magnitude*.
3. **Parameters are not independent in their effect**

### 4.1 · Free vs fixed parameters

You are not obliged to fit all five (see the frontend GUI).

```python
@dataclass(frozen=True)
class ParamSpec:
    name: str      # one of PARAM_NAMES
    free: bool     # True → fitted, False → held exactly
    value: float   # initial guess if free, else the fixed value
    lower: float   # bounds in natural (linear) units
    upper: float
    log: bool      # fitted in log10 space?
```

If you independently know a parameter, fixing it collapses that direction of the search entirely
and makes the rest easier to determine.

In experiment if measured $R_s$ with a four-point probe then fix this in the GUI.

In [33]:
fixed_r_s = 0.4200000000001  # deliberately awkward

specs_fx = default_specs(
    "light", free={"j_ph", "j_0", "n", "r_sh"},   # note: r_s NOT free
    initial={"r_s": fixed_r_s},
)
res_fx = fit_diode(DS.voltage, DS.current, TEMP_K, specs_fx, kind="light")

print(f"fitted      : {', '.join(res_fx.free_names)}")
print(f"Rs supplied : {fixed_r_s!r}")
print(f"Rs returned : {res_fx.params.r_s!r}")
print(f"bitwise identical: {res_fx.params.r_s == fixed_r_s}")

theta0, lower, upper, free_names = pack(specs_fx)
print(f"\noptimiser only ever sees {len(free_names)} numbers: {free_names}")
print("r_s is not among them, so it physically cannot drift.")

fitted      : j_ph, j_0, n, r_sh
Rs supplied : 0.4200000000001
Rs returned : 0.4200000000001
bitwise identical: True

optimiser only ever sees 4 numbers: ('j_ph', 'j_0', 'n', 'r_sh')
r_s is not among them, so it physically cannot drift.


### 4.2 · Why $J_0$ and $R_{sh}$ are fitted in log space

`LOG_PARAMS = {"j_0", "r_sh"}` - these two are fitted as $\log_{10}$ of their value. $J_{ph}$, $n$
and $R_s$ are fitted linearly.

**The intuition.** $J_0$ is "about $10^{-11}$". A step of $10^{-12}$ is problematic when $J_0 = 10^{-13}$. Thus makes sense to step in **ratio** and logarithm makes most sense here.

The Jacobian $\partial J_i/\partial\theta_j$
describe how much the curve moves when nudge each parameter. If one column is $10^{11}$ times bigger
than another, the small one is invisible. Thus, no size of step is suitable for either parameter.

In [34]:
def jacobian_columns(use_log: bool):
    """Central-difference Jacobian columns at a representative optimum."""
    true = DiodeParams(j_ph=0.036, j_0=1e-13, n=1.1, r_s=0.5, r_sh=2500.0, temp_k=298.15)
    vv = np.linspace(0, 0.62, 80)
    base = np.array([true.j_ph, true.j_0, true.n, true.r_s, true.r_sh])
    cols = []
    for i, name in enumerate(PARAM_NAMES):
        lg = use_log and name in ("j_0", "r_sh")
        x = np.log10(base[i]) if lg else base[i]
        h = 1e-6 * max(abs(x), 1.0)
        up, dn = base.copy(), base.copy()
        up[i] = 10 ** (x + h) if lg else x + h
        dn[i] = 10 ** (x - h) if lg else x - h
        f = lambda pv: solve_current(vv, DiodeParams(*pv, temp_k=298.15))
        cols.append((f(up) - f(dn)) / (2 * h))
    return np.array(cols).T


for use_log in (False, True):
    J = jacobian_columns(use_log)
    norms = np.linalg.norm(J, axis=0)
    s = np.linalg.svd(J, compute_uv=False)
    label = "WITH log10 on J0, Rsh" if use_log else "all-linear (no log)"
    print(f"{label}")
    print("   column norms: " + "  ".join(f"{n}={v:.2e}" for n, v in zip(PARAM_NAMES, norms)))
    print(f"   spread (max/min) = {norms.max() / norms.min():.2e}")
    print(f"   cond(J)          = {s.max() / s.min():.2e}\n")

all-linear (no log)
   column norms: j_ph=8.94e+00  j_0=8.65e+04  n=1.93e-02  r_s=1.24e-03  r_sh=5.35e-07
   spread (max/min) = 1.62e+11
   cond(J)          = 4.65e+11

WITH log10 on J0, Rsh
   column norms: j_ph=8.94e+00  j_0=2.20e-03  n=1.93e-02  r_s=1.24e-03  r_sh=3.08e-03
   spread (max/min) = 7.23e+03
   cond(J)          = 6.74e+06



Not everything is logged:

- **$R_s$ can  be zero**, and $\log(0) = -\infty$. §2 proved the model is perfectly
  continuous through $R_s = 0$/
- **$n$ spans ~1–2** and **$J_{ph}$ is known to a few percent** from $J_{sc}$.

### 4.3 · Bounds, and a one-line clamp that prevents a crash

Every parameter carries a box constraint, transformed into fit space alongside its value. Bounds do
two jobs: keep the search where the forward model behaves, and encode physical prior knowledge.

Note this is currently implemented as hard walls only. There is no soft penalty term, so inside
the box every value is treated as equally plausible. This should be changed in the second implementation.

In [35]:
print("DEFAULT_BOUNDS (natural units)      UI slider range        comment")
ui_ranges = {
    "j_ph": ("0 – 50 mA/cm²", "bound is 2x the slider top"),
    "j_0": ("1e-16 – 1e-5", "generous; unlikely to bind"),
    "n": ("1.0 – 2.0", "<-- bound allows n < 1 (unphysical!)"),
    "r_s": ("0 – 5", "zero is legitimate"),
    "r_sh": ("100 – 1e5", "upper bound ~ 'no shunt at all'"),
}
for name in PARAM_NAMES:
    lo, hi = DEFAULT_BOUNDS[name]
    ui, note = ui_ranges[name]
    print(f"  {name:5s} ({lo:g}, {hi:g})".ljust(38) + f"{ui:22s} {note}")

DEFAULT_BOUNDS (natural units)      UI slider range        comment
  j_ph  (0, 0.1)                      0 – 50 mA/cm²          bound is 2x the slider top
  j_0   (1e-20, 0.001)                1e-16 – 1e-5           generous; unlikely to bind
  n     (0.8, 5)                      1.0 – 2.0              <-- bound allows n < 1 (unphysical!)
  r_s   (0, 50)                       0 – 5                  zero is legitimate
  r_sh  (10, 1e+08)                   100 – 1e5              upper bound ~ 'no shunt at all'


Note:
```python
theta0_arr = np.clip(np.asarray(theta0, dtype=float), lower_arr, upper_arr)
```

`least_squares` raises `ValueError` if the starting point $x_0$ is outside the bounds. This would clash with UI so overriden.

In [36]:
# Seed j_0 at 1e-2, far above its upper bound of 1e-3.
specs_bad = default_specs("light", free={"j_0", "n"}, initial={"j_0": 1e-2})
theta0, lower, upper, names = pack(specs_bad)

print(f"free parameters      : {names}")
print(f"seed j_0 requested   : 1e-02  (log10 = {np.log10(1e-2):+.3f})")
print(f"upper bound on j_0   : {DEFAULT_BOUNDS['j_0'][1]:.0e}  (log10 = {upper[0]:+.3f})")
print(f"seed after clamping  : log10 = {theta0[0]:+.3f}  -> j_0 = {10 ** theta0[0]:.1e}")
print(f"\nstart is now feasible: {bool(np.all((theta0 >= lower) & (theta0 <= upper)))}")
print("Without the clamp this exact call would raise ValueError.")

free parameters      : ('j_0', 'n')
seed j_0 requested   : 1e-02  (log10 = -2.000)
upper bound on j_0   : 1e-03  (log10 = -3.000)
seed after clamping  : log10 = -3.000  -> j_0 = 1.0e-03

start is now feasible: True
Without the clamp this exact call would raise ValueError.


### 4.4 · Linear vs log residuals — the most consequential decision

The residual is "how wrong is the model at point $i$". There are two sensible definitions and
they are not interchangeable:

$$ r_i^{\text{lin}} = J^{\text{model}}_i - J^{\text{meas}}_i
   \qquad\qquad
   r_i^{\text{log}} = \log_{10}\!|J^{\text{model}}_i| - \log_{10}\!|J^{\text{meas}}_i| $$

`resolve_residual_space("auto", kind)` picks **log for dark data** and **linear for light**.

**Why light data must be linear.** A light curve crosses zero at $V_{oc}$. Log not valid.

In [37]:
dk = DARK_JV_EXAMPLE
mag = np.abs(dk.current)
print(f"{dk.label}: {dk.voltage.size} points")
print(f"  |J| spans {mag.min():.3e} to {mag.max():.3e} A/cm^2")
print(f"  that is {np.log10(mag.max() / mag.min()):.1f} DECADES of current")
print(f"\nUnder LINEAR residuals, a 1% error at the top of the curve costs")
print(f"~{mag.max() / mag.min() * 0.01:,.0f}x more than a 100% error at the bottom.")
print("The fit would become a fit to the last half-decade, throwing away the")
print("low-injection region -- exactly where shunt and recombination physics live.")

Example: Dark JV: 127 points
  |J| spans 2.347e-06 to 2.463e-01 A/cm^2
  that is 5.0 DECADES of current

Under LINEAR residuals, a 1% error at the top of the curve costs
~1,050x more than a 100% error at the bottom.
The fit would become a fit to the last half-decade, throwing away the
low-injection region -- exactly where shunt and recombination physics live.


In [39]:
def fit_dark(space):
    sp = default_specs("dark", free={"j_0", "n", "r_s", "r_sh"})
    return fit_diode(dk.voltage, dk.current, 298.15, sp, kind="dark", residual_space=space)


r_log, r_lin = fit_dark("log"), fit_dark("linear")

print(f"{'':16s} {'J0':>12s} {'n':>7s} {'Rs':>7s} {'Rsh':>8s} {'linear R2':>11s} {'worst log err':>15s}")
for tag, r in (("log space", r_log), ("linear space", r_lin)):
    p = r.params
    ld = np.log10(np.maximum(np.abs(r.model_current), 1e-9)) - np.log10(np.abs(dk.current))
    print(f"  {tag:14s} {p.j_0:12.3e} {p.n:7.3f} {p.r_s:7.3f} {p.r_sh:8.1f} "
          f"{r.r_squared:11.5f} {np.max(np.abs(ld)):11.3f} dec")

k = int(np.argmax(np.abs(np.log10(np.maximum(np.abs(r_lin.model_current), 1e-9))
                        - np.log10(np.abs(dk.current)))))
print(f"\nWorst point of the linear-space fit: V = {dk.voltage[k]:.4f} V")
print(f"  measured {dk.current[k]:.3e}   model {r_lin.model_current[k]:.3e} A/cm^2")
print(f"  -> wrong by a factor of {abs(r_lin.model_current[k] / dk.current[k]):.2f}")

                           J0       n      Rs      Rsh   linear R2   worst log err
  log space         3.717e-11   1.172   0.192    439.4     0.99962       0.019 dec
  linear space      2.682e-10   1.298   0.154    738.2     0.99997       0.245 dec

Worst point of the linear-space fit: V = 0.0013 V
  measured -3.093e-06   model -1.761e-06 A/cm^2
  -> wrong by a factor of 0.57


**Above result is misleading and actually gives wrong result.** The linear-space fit has the **better R²**
(0.99997 vs 0.99962). It is nonetheless wrong by 0.245 decades - a factor of 1.76 - at low voltage.

Plot on semi-log for reasoning:

In [40]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=dk.voltage, y=np.abs(dk.current), mode="markers",
                         name="measured (5 decades)", marker=dict(color="#444", size=4)))
fig.add_trace(go.Scatter(x=dk.voltage, y=np.abs(r_log.model_current), mode="lines",
                         name=f"log-space fit — rmse_log = {r_log.rmse_log:.3f} dec",
                         line=dict(color="#2563eb", width=2.5)))
fig.add_trace(go.Scatter(x=dk.voltage, y=np.abs(r_lin.model_current), mode="lines",
                         name=f"linear-space fit — R² = {r_lin.r_squared:.5f}",
                         line=dict(color="#dc2626", width=2.5, dash="dash")))
fig.update_layout(
    title="Diff residuals",
    xaxis_title="Voltage (V)", yaxis_title="|J| (A/cm²)",
    yaxis=dict(type="log"), height=460, margin=dict(l=60, r=30, t=60, b=50),
    legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.75)"),
)
fig.show()

The red curve tracks the high-current region beautifully — that is what earns it R² = 0.99997 — then
peels away below ~$10^{-3}$ A/cm², where **four of the five decades of measurement are**.

Two further details:

- **`CURRENT_FLOOR` ($10^{-9}$ A/cm²) is a safety net, not a modelling choice.** It sits three
  decades below the smallest measured dark current, so it never engages here. It exists so a *trial*
  parameter set producing near-zero current cannot generate $-\infty$.
- **The log residual discards sign** — it compares $|J|$, so a model curve of the wrong sign is
  invisible to it. That is safe *only* because `data_import.normalize` has already forced dark data
  to $-|J|$, matching the forward model's convention. Those two decisions are load-bearing for each
  other and they live in **different files**.

Here is the module's own residual plot — the same one the app shows — for the log-space fit:

In [18]:
residual_figure(dk.voltage, r_log.residual, residual_space=r_log.residual_space).show()
print("Structureless scatter about zero = good fit.")
print("Systematic curvature would mean the model itself is wrong (see §6, A1).")

Structureless scatter about zero = good fit.
Systematic curvature would mean the model itself is wrong (see §6, A1).


### 4.5 · Dark data never gets a photocurrent

A dark measurement is taken with the lights off, so $J_{ph} = 0$ **by definition**. Fitting it would
mean fitting a quantity known in advance to be zero — and worse, a nearly-flat offset that could
quietly absorb misfit from anywhere else in the curve.

`default_specs` enforces this *structurally*, three times over: it drops `j_ph` from the free set,
injects it fixed at 0, and pins its bounds to $[0, 0]$. A caller who explicitly asks to fit it does
not get an error — they get the physically correct behaviour, silently.

This matters because the UI's checkbox state persists: tick $J_{ph}$ with light data loaded, switch
to dark, and the stale tick must not leak through.

In [19]:
sneaky = default_specs("dark", free={"j_ph", "j_0", "n"})  # explicitly demanding j_ph
s = sneaky["j_ph"]
print("Asked to fit j_ph on dark data. Got:")
print(f"  free   = {s.free}")
print(f"  value  = {s.value}")
print(f"  bounds = ({s.lower}, {s.upper})")

r = fit_diode(dk.voltage, dk.current, 298.15, sneaky, kind="dark")
print(f"\nactually fitted : {r.free_names}")
print(f"j_ph in result  : {r.params.j_ph}  <- never introduced")

Asked to fit j_ph on dark data. Got:
  free   = False
  value  = 0.0
  bounds = (0.0, 0.0)

actually fitted : ('j_0', 'n')
j_ph in result  : 0.0  <- never introduced


### 4.6 · The optimiser call

```python
result = least_squares(
    _make_residual(voltage, current, specs, temp_k, space, penalty),
    theta0, bounds=(lower, upper),
    method="trf", x_scale="jac", loss=loss, max_nfev=max_nfev,
)
```

| Argument | Why |
|---|---|
| `method="trf"` | Trust Region Reflective — the only SciPy least-squares method that supports **bounds** (`lm` does not). Given §4.3, this follows. |
| `x_scale="jac"` | Rescales variables by the Jacobian each iteration, so the trust region is roughly spherical in scaled space. Second line of defence on conditioning after §4.2. |
| `loss="linear"` | Ordinary least squares — optimal under Gaussian noise, but **no outlier protection**. `soft_l1`/`huber` are plumbed through but the UI never sets them (§6, Gap 6). |
| `jac` *(not passed)* | Defaults to 2-point finite differences: one extra `solve_current` per free parameter per Jacobian. Cheap here, and it avoids hand-deriving $\partial J/\partial\theta$ through a Lambert W. |

Two smaller touches worth noticing.

**The all-fixed shortcut.** If nothing is free, the module skips the optimiser, evaluates once, and
returns `success=True`. That is not just a guard against a SciPy error — it makes "Fit" with every
box unticked mean *"score my current slider values against this data"*, which is a genuinely useful
thing to want:

In [20]:
best = result.params   # the §3 fit
all_fixed = default_specs(
    "light", free=set(),
    initial={"j_ph": best.j_ph, "j_0": best.j_0, "n": best.n,
             "r_s": best.r_s, "r_sh": best.r_sh},
)
r_fixed = fit_diode(DS.voltage, DS.current, TEMP_K, all_fixed, kind="light")
print(f"free_names : {r_fixed.free_names}   (nothing to optimise)")
print(f"success    : {r_fixed.success}")
print(f"message    : {r_fixed.message}")
print(f"RMSE       : {r_fixed.rmse * 1e3:.4f} mA/cm^2  <- still scored against the data")

free_names : ()   (nothing to optimise)
success    : True
message    : No free parameters; evaluated fixed model.
RMSE       : 0.0452 mA/cm^2  <- still scored against the data


**The final re-evaluation.** After the optimiser returns, the module recomputes `solve_current`
cleanly at the solution rather than reusing whatever the residual closure last produced (which might
have been a penalty vector), and wraps *that* in a try/except too. Combined with the docstring's
promise that it "never raises on optimizer failure", the UI can call `fit_diode` with no try/except
and trust it to always return a reportable `FitResult`.

### 4.7 · The penalty is a crash guard, not a barrier

Any forward-model failure inside the residual is trapped and replaced with a large constant
(`PENALTY = 1e6`), preserving the vector length that `least_squares` requires.

It is tempting to read "large penalty" as *repelling* the optimiser. It does not. The penalty is a
**constant**, so its gradient is exactly zero — a perfectly flat plateau with no downhill direction
anywhere on it:

In [21]:
v5 = np.linspace(0.01, 0.6, 20)
j5 = np.full_like(v5, -1e-4)
sp5 = default_specs("dark", free={"j_0", "n"})

original = fitting_module.solve_current
try:
    fitting_module.solve_current = lambda *a, **k: (_ for _ in ()).throw(OverflowError("boom"))
    res_fn = fitting_module._make_residual(v5, j5, sp5, 298.15, "log", fitting_module.PENALTY)
    th, *_ = pack(sp5)
    a = res_fn(th)
    b = res_fn(th + np.array([0.5, 0.2]))
finally:
    fitting_module.solve_current = original   # always restore

print(f"residual at theta0        : {a[:3]}")
print(f"residual at theta0 + delta: {b[:3]}")
print(f"bitwise identical         : {np.array_equal(a, b)}")
print("\n=> the finite-difference Jacobian here is exactly ZERO.")
print("   The penalty stops a crash; it cannot steer the optimiser back to safety.")
print("   The BOUNDS (4.3) are what actually keep it out of trouble.")

residual at theta0        : [1000000. 1000000. 1000000.]
residual at theta0 + delta: [1000000. 1000000. 1000000.]
bitwise identical         : True

=> the finite-difference Jacobian here is exactly ZERO.
   The penalty stops a crash; it cannot steer the optimiser back to safety.
   The BOUNDS (4.3) are what actually keep it out of trouble.


So is the penalty ever reached? The plausible trigger is $e^{b}$ overflowing, which needs
$b > 709.8$. Maximising $b$ over the corners of `DEFAULT_BOUNDS`:

In [22]:
vt = thermal_voltage(298.15)
worst = max(
    ((10 ** 8) * (rs * (0.1 + 1e-3) + V)) / (n * vt * (rs + 10 ** 8))
    for n in (0.8, 1.0, 2.0, 5.0) for V in (0.7, 1.2, 2.0) for rs in (50.0, 0.5)
)
print(f"Vt(298.15 K)         = {vt * 1e3:.4f} mV")
print(f"max b over the box   = {worst:.1f}")
print(f"exp() overflows at   = 709.8")
print(f"headroom             = {709.8 / worst:.1f}x")
print("\nSo on the current bounds the overflow path is effectively unreachable.")
print("The try/except is cheap insurance against FUTURE changes -- widening the n")
print("lower bound or the voltage range would erode that margin quietly.")

Vt(298.15 K)         = 25.6926 mV
max b over the box   = 343.0
exp() overflows at   = 709.8
headroom             = 2.1x

So on the current bounds the overflow path is effectively unreachable.
The try/except is cheap insurance against FUTURE changes -- widening the n
lower bound or the voltage range would erode that margin quietly.


### 4.8 · What gets reported

`FitResult` reports RMSE, R² and max-absolute-residual **always in linear current units**, whatever
space the fit ran in, plus `rmse_log` when log residuals were used.

$$ \mathrm{RMSE} = \sqrt{\tfrac1m\textstyle\sum_i (J^{\text{model}}_i - J^{\text{meas}}_i)^2},
   \qquad
   R^2 = 1 - \frac{\sum_i (J^{\text{model}}_i - J^{\text{meas}}_i)^2}
                  {\sum_i (J^{\text{meas}}_i - \bar{J}^{\text{meas}})^2} $$

Reporting headline metrics in a **fixed** space is the right call: it makes a log-space dark fit and
a linear-space light fit comparable, and the number in the UI is in the same mA/cm² you read off the
graph. Decoupling the *reported* metric from the *optimised* objective is a deliberate separation.

The docstring's warning that "a linear R² on dark data reads deceptively close to 1" is not hedging —
§4.4 produced a fit with R² = 0.99997 that is 44% wrong across most of its range. **The honest way to
read a dark fit is: ignore R², read `rmse_log`.** In the log-space fit that was 0.005 decades ≈ 1.2%
mean relative error — a genuinely informative number, where R² = 0.9996 tells you nothing.

---
## 5 · Degeneracy — the punchline

The README calls this the core technical challenge of the whole project:

> *Naive fitting produces good numerical agreement with **parameter degeneracy** — many different
> parameter combinations fit equally well, but most of them are physically meaningless.*

Let us find out exactly how bad it is, on the repo's own data. The picture turns out to be sharper
than that framing suggests — and the sharpness is useful.

### 5.1 · First, the reassuring result

If the problem were badly behaved, different starting guesses would land in different places. Try
four, spanning six decades of $J_0$:

In [23]:
starts = [
    {"j_ph": 0.035, "j_0": 1e-12, "n": 1.0, "r_s": 0.5, "r_sh": 1000.0},
    {"j_ph": 0.036, "j_0": 1e-14, "n": 1.0, "r_s": 0.1, "r_sh": 5000.0},
    {"j_ph": 0.037, "j_0": 1e-10, "n": 1.8, "r_s": 2.0, "r_sh": 300.0},
    {"j_ph": 0.036, "j_0": 1e-16, "n": 1.2, "r_s": 3.0, "r_sh": 50000.0},
]
print(f"{'start':>6s} {'Jph':>10s} {'J0':>12s} {'n':>8s} {'Rs':>8s} {'Rsh':>9s} {'RMSE':>11s}")
for i, s in enumerate(starts, 1):
    sp = default_specs("light", free=set(PARAM_NAMES), initial=s)
    r = fit_diode(DS.voltage, DS.current, TEMP_K, sp, kind="light")
    q = r.params
    print(f"{i:>6d} {q.j_ph:10.6f} {q.j_0:12.4e} {q.n:8.4f} {q.r_s:8.4f} {q.r_sh:9.2f} {r.rmse:11.4e}")

print("\nIdentical to ~5 significant figures, from wildly different starts.")
print("The optimiser is NOT fragile, and there is no multi-start problem on clean data.")

 start        Jph           J0        n       Rs       Rsh        RMSE
     1   0.036093   4.5802e-11   1.1669   0.2932    437.18  4.5236e-05
     2   0.036093   4.5802e-11   1.1669   0.2932    437.18  4.5236e-05
     3   0.036093   4.5802e-11   1.1669   0.2932    437.18  4.5236e-05
     4   0.036093   4.5802e-11   1.1669   0.2932    437.18  4.5236e-05

Identical to ~5 significant figures, from wildly different starts.
The optimiser is NOT fragile, and there is no multi-start problem on clean data.


And an independent cross-check: the light and dark examples are different measurements, fitted with
different residual spaces and different free sets. Do they agree about the cell?

In [24]:
sp_l = default_specs("light", free=set(PARAM_NAMES))
r_l = fit_diode(LIGHT_JV_EXAMPLE.voltage, LIGHT_JV_EXAMPLE.current, 298.15, sp_l, kind="light")

print(f"{'param':>6s} {'light fit':>13s} {'dark fit':>13s} {'agreement':>12s}")
for nm, unit in (("j_0", ""), ("n", ""), ("r_s", ""), ("r_sh", "")):
    a, b = getattr(r_l.params, nm), getattr(r_log.params, nm)
    print(f"{nm:>6s} {a:13.5g} {b:13.5g} {abs(a / b):11.3f}x")

print("\nn and Rsh agree to better than 1% across two independent datasets.")
print("That is a strong end-to-end validation: forward model, sign conventions,")
print("unit handling and BOTH residual spaces all check out.")
print("(It also strongly suggests both examples are the same PV Lighthouse cell.)")
print("Note Rs agrees worst -- that is not noise, and 5.3 explains why.")

 param     light fit      dark fit    agreement
   j_0    4.5802e-11    3.7174e-11       1.232x
     n        1.1669        1.1718       0.996x
   r_s       0.29321       0.19227       1.525x
  r_sh        437.18        439.36       0.995x

n and Rsh agree to better than 1% across two independent datasets.
That is a strong end-to-end validation: forward model, sign conventions,
unit handling and BOTH residual spaces all check out.
(It also strongly suggests both examples are the same PV Lighthouse cell.)
Note Rs agrees worst -- that is not noise, and 5.3 explains why.


### 5.2 · Now the problem

Real measurements have noise. Add a modest 0.5% of $J_{sc}$ — optimistic for a real instrument — and
refit twelve times, redrawing **only the noise** each time. The cell never changes:

In [25]:
rng = np.random.default_rng(7)
rows = []
for _ in range(12):
    noisy = DS.current + rng.normal(0, 0.005 * np.max(np.abs(DS.current)), DS.current.shape)
    sp = default_specs("light", free=set(PARAM_NAMES))
    r = fit_diode(DS.voltage, noisy, TEMP_K, sp, kind="light")
    q = r.params
    rows.append([q.j_ph, q.j_0, q.n, q.r_s, q.r_sh, r.rmse])
A = np.array(rows)

print(f"{'param':>6s} {'min':>12s} {'median':>12s} {'max':>12s} {'spread':>9s} {'CV':>8s}")
for i, nm in enumerate(PARAM_NAMES):
    c = A[:, i]
    print(f"{nm:>6s} {c.min():12.4e} {np.median(c):12.4e} {c.max():12.4e} "
          f"{c.max() / c.min():8.2f}x {c.std() / abs(c.mean()) * 100:7.2f}%")
print(f"\nRMSE across all 12: {A[:, 5].min():.3e} to {A[:, 5].max():.3e} A/cm^2")
print("-- i.e. every one of these is a 'good' fit by any metric the module reports.")

 param          min       median          max    spread       CV
  j_ph   3.6022e-02   3.6104e-02   3.6139e-02     1.00x    0.09%
   j_0   1.5793e-11   3.1656e-11   8.6180e-11     5.46x   50.39%
     n   1.1090e+00   1.1462e+00   1.2041e+00     1.09x    2.07%
   r_s   2.4318e-01   3.1973e-01   3.5200e-01     1.45x    9.10%
  r_sh   3.9939e+02   4.1930e+02   4.4585e+02     1.12x    2.96%

RMSE across all 12: 1.439e-04 to 1.900e-04 A/cm^2
-- i.e. every one of these is a 'good' fit by any metric the module reports.


Look at $J_0$: it ranges over a factor of ~5.5 (CV ≈ 50%) while $J_{ph}$ barely moves (CV 0.09%).

**The degeneracy is not that the optimiser fails. It is that it succeeds — confidently — at
slightly different answers.** A single fit reporting $J_0 = 3.2\times10^{-11}$ with R² = 0.9999
gives you no hint that $1.6\times10^{-11}$ and $8.6\times10^{-11}$ fit the same data just as well.

### 5.3 · Why: the $V_{oc}$-preserving ridge

Build the covariance from the Jacobian at the optimum, $\Sigma = \sigma^2(\mathbf{J}^\top\mathbf{J})^{-1}$,
and normalise it to a correlation matrix. This is standard, costs no extra forward evaluations —
and the module currently throws away the `result.jac` needed to do it (§6, Gap 2).

In [26]:
opt = np.array([r_l.params.j_ph, np.log10(r_l.params.j_0), r_l.params.n,
                r_l.params.r_s, np.log10(r_l.params.r_sh)])
sp_j = default_specs("light", free=set(PARAM_NAMES))
res_fn = fitting_module._make_residual(DS.voltage, DS.current, sp_j, TEMP_K, "linear", 1e6)

m = DS.voltage.size
Jm = np.zeros((m, 5))
for i in range(5):
    h = 1e-7 * max(abs(opt[i]), 1.0)
    up, dn = opt.copy(), opt.copy()
    up[i] += h
    dn[i] -= h
    Jm[:, i] = (res_fn(up) - res_fn(dn)) / (2 * h)

resid = res_fn(opt)
s2 = float(resid @ resid) / (m - 5)
cov = s2 * np.linalg.pinv(Jm.T @ Jm)
sd = np.sqrt(np.diag(cov))
corr = cov / np.outer(sd, sd)

labels = ["j_ph", "log10 J0", "n", "r_s", "log10 Rsh"]
print("1-sigma standard errors (NOT currently reported by FitResult):")
for nm, v, e in zip(labels, opt, sd):
    print(f"   {nm:10s} = {v:11.5f}  +/- {e:.5f}   ({abs(e / v) * 100:5.2f}%)")

print("\nCorrelation matrix:")
print("            " + "".join(f"{n:>11s}" for n in labels))
for i, n1 in enumerate(labels):
    print(f"  {n1:10s}" + "".join(f"{corr[i, j]:11.3f}" for j in range(5)))

w = np.linalg.eigvalsh(Jm.T @ Jm)
print(f"\neigenvalue spread of J^T J = {w.max() / w.min():.2e}")
print(f"sloppiest vs stiffest direction = {np.sqrt(w.max() / w.min()):.2e}x")

1-sigma standard errors (NOT currently reported by FitResult):
   j_ph       =     0.03609  +/- 0.00001   ( 0.04%)
   log10 J0   =   -10.33912  +/- 0.07262   ( 0.70%)
   n          =     1.16688  +/- 0.00950   ( 0.81%)
   r_s        =     0.29321  +/- 0.01218   ( 4.15%)
   log10 Rsh  =     2.64066  +/- 0.00981   ( 0.37%)

Correlation matrix:
                   j_ph   log10 J0          n        r_s  log10 Rsh
  j_ph            1.000     -0.398     -0.397      0.316     -0.884
  log10 J0       -0.398      1.000      1.000     -0.952      0.551
  n              -0.397      1.000      1.000     -0.950      0.549
  r_s             0.316     -0.952     -0.950      1.000     -0.420
  log10 Rsh      -0.884      0.551      0.549     -0.420      1.000

eigenvalue spread of J^T J = 1.53e+08
sloppiest vs stiffest direction = 1.24e+04x


**`corr(log J₀, n) = 1.000`**, to three decimal places. These two parameters are not separately
identifiable from this data *at all* — only one combination of them is. (And `corr(log J₀, Rs) ≈ −0.95`.)

Why? Look at what happens at $V_{oc}$. By definition $J = 0$ there, so the implicit equation reads

$$ 0 = J_{ph} - J_0\left(e^{\frac{V_{oc}}{nV_t}} - 1\right) - \frac{V_{oc}}{R_{sh}} $$

Neglect the shunt leak and the $-1$ (both small at $V_{oc}$) and rearrange:

$$ \boxed{\;J_0 \;\approx\; J_{ph}\,e^{-\frac{V_{oc}}{nV_t}}\;}
   \qquad\Longleftrightarrow\qquad
   \ln J_0 \;\approx\; \ln J_{ph} \;-\; \frac{V_{oc}}{V_t}\cdot\frac{1}{n} $$

The data pins $J_{ph}$ (from $J_{sc}$) and $V_{oc}$ (from the zero crossing) very tightly — which is
exactly why $J_{ph}$ had the smallest CV in §5.2. But that leaves $n$ and $\ln J_0$ free to **slide
together** along a straight line of slope $-V_{oc}/V_t$, holding $V_{oc}$ fixed and the curve
essentially unchanged.

**That line is the ridge.** Test it by *profiling*: fix $n$ at each of 46 values and re-optimise all
the other parameters to convergence.

In [27]:
n_grid = np.linspace(1.00, 1.45, 46)
prof = []
for nn in n_grid:
    sp = default_specs(
        "light", free={"j_ph", "j_0", "r_s", "r_sh"},
        initial={"j_ph": r_l.params.j_ph, "j_0": r_l.params.j_0, "n": nn,
                 "r_s": r_l.params.r_s, "r_sh": r_l.params.r_sh},
    )
    r = fit_diode(DS.voltage, DS.current, TEMP_K, sp, kind="light")
    prof.append((nn, r.params.j_0, r.rmse, r.params.r_s))
prof = np.array(prof)

voc = key_metrics(DS.voltage, DS.current)["voc"]
jph = r_l.params.j_ph
vt = thermal_voltage(TEMP_K)

# Is ln J0 really linear in 1/n, with the predicted slope?
x, y = 1.0 / prof[:, 0], np.log(prof[:, 1])
slope, intercept = np.polyfit(x, y, 1)
r2 = 1 - np.sum((y - np.polyval([slope, intercept], x)) ** 2) / np.sum((y - y.mean()) ** 2)

print(f"Regression of ln J0 on 1/n across the profile:")
print(f"  slope            = {slope:.4f}")
print(f"  R^2              = {r2:.8f}      <-- essentially a perfect straight line")
print(f"\n  predicted slope -Voc/Vt = {-voc / vt:.4f}")
print(f"  implied Voc  = {-slope * vt:.4f} V")
print(f"  measured Voc = {voc:.4f} V   ({abs(-slope * vt - voc) / voc * 100:.1f}% -- the neglected shunt)")
print(f"\nJ0 spans {prof[:, 1].min():.3e} -> {prof[:, 1].max():.3e} "
      f"= {np.log10(prof[:, 1].max() / prof[:, 1].min()):.2f} DECADES")
print(f"...while RMSE never exceeds {prof[:, 2].max() / r_l.rmse:.2f}x its best value.")

Regression of ln J0 on 1/n across the profile:
  slope            = -23.9734
  R^2              = 0.99999992      <-- essentially a perfect straight line

  predicted slope -Voc/Vt = -23.8474
  implied Voc  = 0.6159 V
  measured Voc = 0.6127 V   (0.5% -- the neglected shunt)

J0 spans 1.485e-12 -> 2.522e-09 = 3.23 DECADES
...while RMSE never exceeds 4.00x its best value.


**R² = 0.9999999.** The ridge is not an artefact or a numerical accident — it is the exact statement
that *the fit is free to trade $n$ against $J_0$ however it likes, provided it reproduces $V_{oc}$*.

Now watch it happen. Drag $n$: the dot is what the optimiser picks for $J_0$ when you *force* that
$n$, the dashed line is the law $J_0 = J_{ph}e^{-V_{oc}/nV_t}$ — **derived above, no fitting, no free
constants** — and the right panel is what it costs you.

In [28]:
law_n = np.linspace(1.0, 1.45, 200)
law_j0 = jph * np.exp(-voc / (law_n * vt))
NOISE = 0.005 * 0.036   # the 0.5% noise level from 5.2, in RMSE terms


def ridge_view(n_choice=1.167):
    i = int(np.argmin(np.abs(prof[:, 0] - n_choice)))
    n_i, j0_i, rmse_i, rs_i = prof[i]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=law_n, y=law_j0, mode="lines", name="law: Jph·exp(−Voc/nVt)",
                             line=dict(color="#dc2626", width=3, dash="dash")))
    fig.add_trace(go.Scatter(x=prof[:, 0], y=prof[:, 1], mode="markers", name="profiled refit",
                             marker=dict(color="#2563eb", size=5)))
    fig.add_trace(go.Scatter(x=[n_i], y=[j0_i], mode="markers", name="your n",
                             marker=dict(color="#111", size=13, symbol="circle-open",
                                         line=dict(width=3))))
    fig.update_layout(
        title=(f"n = {n_i:.3f}  ->  J₀ = {j0_i:.3e}   |   "
               f"RMSE = {rmse_i * 1e3:.4f} mA/cm² ({rmse_i / prof[:, 2].min():.2f}× best)   |   "
               f"Rs = {rs_i:.3f}"),
        xaxis_title="ideality factor n", yaxis_title="saturation current J₀ (A/cm²)",
        yaxis=dict(type="log"), height=430, margin=dict(l=60, r=30, t=60, b=50),
        legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.75)"),
    )
    fig.show()

    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=prof[:, 0], y=prof[:, 2] * 1e3, mode="lines",
                              name="RMSE", line=dict(color="#2563eb", width=2.5)))
    fig2.add_hline(y=NOISE * 1e3, line=dict(color="#b45309", width=2, dash="dash"),
                   annotation_text="0.5% measurement noise", annotation_position="top right")
    fig2.add_trace(go.Scatter(x=[n_i], y=[rmse_i * 1e3], mode="markers", name="your n",
                              marker=dict(color="#111", size=13, symbol="circle-open",
                                          line=dict(width=3))))
    fig2.update_layout(
        title="Everything below the dashed line is indistinguishable from the best fit",
        xaxis_title="ideality factor n", yaxis_title="RMSE (mA/cm²)",
        yaxis=dict(range=[0, 0.21]), height=330, margin=dict(l=60, r=30, t=50, b=50),
        showlegend=False,
    )
    fig2.show()


if HAVE_WIDGETS:
    n_slider = widgets.FloatSlider(value=1.167, min=1.0, max=1.45, step=0.01,
                                   description="force n =", readout_format=".3f",
                                   style={"description_width": "90px"},
                                   layout=widgets.Layout(width="520px"))
    display(widgets.VBox([n_slider, widgets.interactive_output(ridge_view, {"n_choice": n_slider})]))
else:
    ridge_view()

The dots sit on the dashed line, and the entire n range stays under the noise floor. **Across
$n \in [1.00, 1.45]$, $J_0$ moves over 3.2 decades while RMSE never exceeds 4× its best — and 0.5%
noise *is* a 4× RMSE band.** Every one of those fits is indistinguishable from the "right" answer.

Notice $R_s$ in the title too: it slides from ~0.50 to 0.00 across the same sweep
($\mathrm{corr}(\log J_0, R_s) = -0.95$). Raising $n$ softens the exponential's turn-on; reducing
$R_s$ sharpens the knee back. They compensate. That is why §5.1's light and dark fits disagreed most
on $R_s$, and why $R_s$ carries the largest relative standard error of any parameter.

> ### The practical consequence
>
> A single-diode fit to a light J–V curve **cannot determine $n$ and $J_0$ separately**, no matter
> how good the optimiser or how high the R². It determines $J_{ph}$, $V_{oc}$, and *one combination*
> of $n$ and $J_0$.
>
> Breaking the degeneracy needs information the light curve does not contain: a **dark curve** (which
> probes the exponential over five decades instead of near $V_{oc}$ alone), a $J_{sc}$–$V_{oc}$ sweep
> across illumination levels, an independent $R_s$ measurement, or a prior on $n$. §4.1's free/fixed
> mechanism is precisely the hook for feeding that in — and §5.1's light-vs-dark agreement on $n$ to
> 0.4% is exactly the kind of cross-constraint that works.
>
> **For Phase C's ten-parameter tandem fits, where two of these ridges exist at once, this is the
> thing that will bite.**

---
## 6 · Assumptions and gaps

### Assumptions `fitting.py` makes

| # | Assumption | If violated |
|---|---|---|
| **A1** | The single-diode model is the right physics for the data | Systematic residual curvature. Not detected automatically — the residual plot is the only signal. |
| **A2** | Temperature is known and fixed, not fitted | $nV_t$ is the only route $T$ takes into the model, so a wrong $T$ is absorbed almost entirely into $n$. |
| **A3** | Input current is already in the model's sign convention | Light fit converges to nonsense; a dark fit in log space cannot even see the error (§4.4). |
| **A4** | Voltages are exact; all error is in current | Least squares stops being the right estimator. Usually fine for J–V, where voltage is the controlled variable. |
| **A5** | Errors are independent, constant variance **in the chosen space** | This is the real content of §4.4: linear assumes constant *absolute* error, log constant *relative* error. |
| **A6** | No outliers | `loss="linear"` lets one bad point dominate. `soft_l1`/`huber` exist but are unreachable from the UI. |
| **A7** | Inside the bounds box, all values are equally plausible | No soft prior exists, so nothing separates *physically sensible* from merely *feasible*. Given §5, this is the assumption the project most needs to relax. |
| **A8** | The optimum is well-identified | **False on this data** (§5). Metrics are reported with no uncertainty, so a degenerate fit looks identical to a determined one. |

### Gaps, ranked by what I would fix first

| | Gap | Severity |
|---|---|---|
| **1** | **The fit temperature ignores the dataset.** `ImportedDataset.temp_k` is documented as "used as a fixed input when fitting" and never is — `pages/01_single_diode.py:513,537` passes the cell-temperature *slider* instead. By A2 the error lands in $n$: fitting 60 °C data at 25 °C biases $n$ by ~12%. Either wire `dataset.temp_k` through or drop the field. | correctness |
| **2** | **No uncertainty or correlation is reported.** Given §5 this is the highest-value addition available. `least_squares` already returns `result.jac` and it is discarded; ~20 lines turn it into standard errors and the correlation matrix — no new dependency, no extra forward evaluations. It turns *"$J_0 = 4.6\times10^{-11}$, R² = 0.99996"* into *"…, 95% CI spanning a decade, 100% correlated with $n$"*, which is the honest answer. | highest value |
| **3** | **The $n$ lower bound admits unphysical diodes.** `DEFAULT_BOUNDS["n"] = (0.8, 5.0)`, but $n < 1$ has no meaning in the Shockley model. It is reachable: force $R_s \geq 1.0$ and the fit pins $n = 0.800$. The UI slider runs 1.0–2.0, so the fit can return a value the control cannot represent. In a project aiming to "penalise unphysical regions heavily", a bound admitting $n = 0.8$ points the wrong way. | physicality |
| **4** | **Fitted parameters are not fed back to the controls.** After a fit the sliders keep their pre-fit values, so the solid "model" trace still shows the *guess*. Possibly deliberate (a fitted value can fall outside a slider's range — see Gap 3), but there is no way to adopt a fit and keep exploring. | UX |
| **5** | **Single start, no multi-start.** §5.1 shows this is fine on the clean example — but that is a property of *this data*, not a guarantee, and Phase C will not inherit it. | robustness |
| **6** | **Robust losses are plumbed but unreachable.** `fit_diode` accepts `loss` and forwards it; no caller ever passes anything but the default. Exposing it next to the existing "Residual space" radio would be nearly free. | minor |
| **7** | **A docstring describes the wrong pipeline order.** `data_import.build_dataset` states "… → `_validate` (in base units) → `to_base_units` → …", listing validation before the conversion it depends on. The code (lines 268–271) does it correctly; only the docstring is out of order. | docs |

Gap 3 is easy to see for yourself — force $R_s$ high and watch the optimiser flee to a corner rather
than tell you the constraint is wrong:

In [29]:
print(f"{'Rs fixed':>9s} {'J0':>12s} {'n':>8s} {'Rsh':>11s} {'RMSE':>11s}   note")
for rs_fixed in (0.0, 0.25, 0.5, 1.0, 2.0, 4.0):
    sp = default_specs(
        "light", free={"j_ph", "j_0", "n", "r_sh"},
        initial={"j_ph": 0.036, "j_0": 1e-13, "n": 1.0, "r_s": rs_fixed, "r_sh": 2000.0},
    )
    r = fit_diode(DS.voltage, DS.current, TEMP_K, sp, kind="light")
    q = r.params
    flags = []
    if abs(q.n - DEFAULT_BOUNDS["n"][0]) < 1e-6:
        flags.append("n PINNED at 0.8 (unphysical)")
    if q.r_sh > 0.9 * DEFAULT_BOUNDS["r_sh"][1]:
        flags.append("Rsh PINNED at upper bound")
    print(f"{rs_fixed:9.2f} {q.j_0:12.4e} {q.n:8.4f} {q.r_sh:11.4g} {r.rmse:11.4e}   "
          + "; ".join(flags))

print("\nAn active bound is diagnostic information -- it means your constraint is wrong.")
print("Right now it is silently accepted and displayed as if it were a result.")

 Rs fixed           J0        n         Rsh        RMSE   note
     0.00   1.2964e-09   1.3934       579.5  1.4614e-04   
     0.25   7.9572e-11   1.1991       452.5  4.9726e-05   
     0.50   2.2775e-12   1.0184       379.5  1.0925e-04   
     1.00   3.3360e-15   0.8000       386.6  4.0515e-04   n PINNED at 0.8 (unphysical)
     2.00   1.7083e-15   0.8000        1372  1.3771e-03   n PINNED at 0.8 (unphysical)
     4.00   2.5127e-16   0.8000   9.993e+07  2.5503e-03   n PINNED at 0.8 (unphysical); Rsh PINNED at upper bound

An active bound is diagnostic information -- it means your constraint is wrong.
Right now it is silently accepted and displayed as if it were a result.


---
## 7 · References

**The code this notebook explains**

| Module | Role |
|---|---|
| `src/models/fitting.py` | the subject: bounds, residuals, penalties, `least_squares` |
| `src/models/single_diode.py` | the forward model — Lambert-W solver, metrics |
| `src/models/data_import.py` | parsing, unit conversion, the dark sign convention |
| `src/models/examples.py` | the two built-in datasets used throughout |
| `ui/plotting.py` | the figure builders reused here |

**Method / literature**

- Jain & Kapoor, *Exact analytical solutions of the parameters of real solar cells using Lambert
  W-function*, Sol. Energy Mater. Sol. Cells — <https://doi.org/10.1016/j.solmat.2003.11.018>
- <https://doi.org/10.1016/j.solmat.2005.04.023>
- <https://doi.org/10.1016/j.rser.2015.11.051>
- <https://doi.org/10.1016/j.solener.2023.111930>
- <https://doi.org/10.1016/j.solener.2023.01.009>
- De Soto et al. (2006) five-parameter model —
  [Sandia PVPMC](https://pvpmc.sandia.gov/modeling-guide/2-dc-module-iv/single-diode-equivalent-circuit-models/de-soto-five-parameter-module-model/)
- [PV Lighthouse equivalent-circuit calculator](https://www.pvlighthouse.com.au/equivalent-circuit) —
  the source of the unit convention and the cross-check reference
- [PVsyst one-diode model](https://www.pvsyst.com/help/physical-models-used/pv-module-standard-one-diode-model/index.html)
- Joule paper — tandem conductivity / unphysicality —
  <https://www.cell.com/joule/abstract/S2542-4351(25)00392-7>

---

*Every number above was computed live by this notebook against `src/models/`. Nothing is quoted from
memory. If a number here disagrees with the code, the code is right and this notebook is stale — so
re-run it.*